In [74]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,SampleID,Text,language
0,S1,สัมประสิทธิ์ฮอลล์ ฟิสิกส์ไฟฟ้า เกี่ยวกับสนามแม...,Thai
1,S2,เกิดวันที่ พฤศจิกายน ภาคอะนิเมะ คดีฆาตกรรมบนจ...,Thai
2,S3,i omgivningarna runt manigotagan river park re...,Swedish
3,S4,நிஞ்சா ஹட்டோரி 忍者ハットリくん ninja hattori என்பது க...,Tamil
4,S5,эта страница деятельности м в ломоносова — ярк...,Russian


In [75]:
from sklearn.feature_extraction.text import TfidfVectorizer

all_texts = pd.concat([df_train.Text, df_test.Text])
vect = TfidfVectorizer(analyzer='char', ngram_range=(3,5)).fit(all_texts)
X = vect.transform(df_train.Text)
X_final = vect.transform(df_test.Text)

X

<17600x3266822 sparse matrix of type '<class 'numpy.float64'>'
	with 14822177 stored elements in Compressed Sparse Row format>

In [76]:
from sklearn.preprocessing import LabelEncoder

enc = LabelEncoder()
y = enc.fit_transform(df_train.language)

y

array([19, 19, 17, ..., 20, 13,  8])

In [77]:
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

model = LinearSVC()

cross_val_score(model, X, y, cv=5, scoring='accuracy')

array([0.98494318, 0.98863636, 0.98721591, 0.98721591, 0.98551136])

In [78]:
model.fit(X, y)
y_preds = model.predict(X_final)

In [79]:
y_preds = enc.inverse_transform(y_preds)
y_preds

array(['Japanese', 'Russian', 'Latin', ..., 'English', 'Pushto', 'Dutch'],
      dtype=object)

In [80]:
submission = pd.DataFrame({
    "SampleID": df_test.SampleID,
    "language": y_preds
})

submission

,SampleID,language
0,S17601,Japanese
1,S17602,Russian
2,S17603,Latin
3,S17604,Pushto
4,S17605,Hindi
...,...,...
4395,S21996,Dutch
4396,S21997,Chinese
4397,S21998,English
4398,S21999,Pushto


In [81]:
submission.to_csv("submission.csv", index=False)